# Track 01 — EXAONE 기본기

**구성** : 각 `Session`은 코드 설명(텍스트) -> 코드 -> 해석(텍스트) 순으로 정리되어 있습니다.

**목표** : Track 01의 목표는 대화·스트리밍·구조화 출력·함수 호출·ThinkingRouter 등 EXAONE 모델 기본기를 익히는 것입니다.

**산출물:** `_out/*.json`, `korean_golden.jsonl`


In [1]:
import json
import logging
import time
import warnings
from pathlib import Path

# (en) ----- setup: paths & client -----
# (kr) ----- setup: 경로와 클라이언트 -----
try:
    import exaone
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "exaone 가 설치되지 않았습니다. pip install -r requirements.txt && pip install -e ."
    ) from exc
for _log_name in ("exaone", "exaone.llm", "exaone.llm.exaone_client", "exaone.context_management", "urllib3"):
    logging.getLogger(_log_name).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Unverified HTTPS request")
exaone.load_project_env()
ROOT = exaone.project_root()
DATA = ROOT / "recipes" / "track01_exaone_foundation" / "data"
out_dir = Path("_out")
out_dir.mkdir(parents=True, exist_ok=True)
client = exaone.integrations.build_llm_from_env()
print("model:", client.model)


# (en) shared helper: truncate long text for printing (… only when actually cut)
# (kr) 공용 헬퍼: 긴 텍스트를 출력용으로 자른다(실제로 잘릴 때만 …)
def _preview(text, n=55):
    return text[:n] + ("…" if len(text) > n else "")

# (en) shared helper: ANSI bold (+ color, default blue) for printing
# (kr) 공용 헬퍼: 출력용 ANSI 볼드(+색, 기본 파랑)
def _bold(text, color="blue"):
    codes = {"black": 30, "red": 31, "green": 32, "yellow": 33, "blue": 34, "cyan": 36}
    prefix = "\033[1m" + (f"\033[{codes[color]}m" if color in codes else "")
    return f"{prefix}{text}\033[0m"

model: LGAI-EXAONE/K-EXAONE-236B-A23B


**출력 해석:** `model:`·경로 변수가 보이면 이 노트북에서 쓸 client·DATA 가 준비된 것입니다.


## Session 1. 대화·스트리밍

**테스트 시나리오** — `data/multi_turn_turns.json` 에 멀티턴 질문(3턴) + 스트리밍 테스트 용 질문(1턴)이 포함되어 있습니다.  
`turn_cfg` 로 질문을 로드하고 **같은 주제의 대화를 이어가며** 1턴 → 멀티턴 → 스트리밍을 각각 수행하여 봅니다. (답변 완성도와 관계없이 기능 작동 여부를 중점적으로 확인) 

| 키 | 사용되는 Session | 질문 내용 |
|---|---|---|
| `turns[0]` | Session 1-2 | 멀티스레딩 vs 멀티프로세싱 (한 줄) |
| `turns[1]` | Session 1-3 | CPU-bound 에 뭐가 나은지 |
| `turns[2]` | Session 1-3 | GIL 이 풀리는 사례 |
| `stream_user` | Session 1-4 | numpy 와 GIL (네 문장, 스트리밍) |

질문 문장은 JSON 에 고정돼 있습니다. 바꾸고 싶으면 `multi_turn_turns.json` 을 수정한 뒤 Session 1-1 부터 다시 실행하세요.


### Session 1-1. turn_cfg 로드

**하는 일:** `turn_cfg` — 멀티턴 질문 3개(`turns`)와 스트리밍용 질문 1개(`stream_user`)를 JSON 에서 읽고, Session 1 전체가 쓸 `OPT` 를 만듭니다.

**입력:** `data/multi_turn_turns.json`

**정상:** `시나리오:` 한 줄 + `턴 수: 3` + 질문 3줄 + `stream_user:` 한 줄

**의미:** 아래 1-2~1-4 가 **같은 turn_cfg** 를 사용합니다. 하위 Session에서 사용될 질문을 확인할 수 있습니다.


In [2]:
# (en) turn_cfg — fixed multi-turn script (Python concurrency); not a pass/fail test.
# (kr) turn_cfg — 파이썬 동시성 주제의 고정 멀티턴 스크립트. 채점 테스트가 아님.
turn_cfg = json.loads((DATA / "multi_turn_turns.json").read_text(encoding="utf-8"))
OPT = exaone.llm.ExaoneGenerateOptions(enable_thinking=False, max_new_tokens=512)

print("시나리오: 파이썬 동시성 주제의 멀티턴 3질문 + 스트리밍 테스트용 1질문 (multi_turn_turns.json)")
print("멀티턴 질문 턴 수:", len(turn_cfg["turns"]))
for i, q in enumerate(turn_cfg["turns"], 1):
    print(f"  turns[{i-1}] turn {i}:", q)
print("스트리밍 질문:")
print("  stream_user :", turn_cfg["stream_user"])


시나리오: 파이썬 동시성 주제의 멀티턴 3질문 + 스트리밍 테스트용 1질문 (multi_turn_turns.json)
멀티턴 질문 턴 수: 3
  turns[0] turn 1: 파이썬 멀티스레딩과 멀티프로세싱 차이를 한 줄로.
  turns[1] turn 2: CPU-bound 에는 어느 쪽이 나은지, 이유도 한 줄로.
  turns[2] turn 3: GIL 이 풀리는 대표 사례 두 가지만.
스트리밍 질문:
  stream_user : numpy 가 GIL 영향을 덜 받는 이유를 네 문장으로.


**출력 해석:** `시나리오:` 와 `turns[0]~[2]`·`stream_user` 질문이 보이면 `turn_cfg` 가 로드된 것입니다. 1-2 는 `turns[0]` 만, 1-3 은 3턴 전부, 1-4 는 `stream_user` 를 씁니다.

### Session 1-2. 1턴 chat()

**하는 일:** `turn_cfg["turns"][0]`("멀티스레딩 vs 멀티프로세싱") 한 건만 `chat()` 으로 보냅니다.

**정상:** `질문:` 이 turns[0] 과 같고, `답:` 과 `prompt_tokens` 숫자가 출력됩니다.

**의미:** 앞 대화가 전혀 없는 가장 단순한 1턴 baseline 입니다. 여기서 본 `prompt_tokens` 가 이후 멀티턴에서 얼마나 늘어나는지 비교 기준이 됩니다.

In [3]:
messages = [exaone.llm.ExaoneMessage(role="user", content=turn_cfg["turns"][0])]
resp = client.chat(messages, options=OPT)
pt = (resp.usage or {}).get("prompt_tokens")
print("질문:", turn_cfg["turns"][0])
print("답:", resp.content)
print("prompt_tokens:", pt)


질문: 파이썬 멀티스레딩과 멀티프로세싱 차이를 한 줄로.
답: 멀티스레딩은 한 프로세스 내에서 여러 스레드가 공유 메모리로 협업하며 경량화된 컨텍스트 스위칭으로 동작하고, 멀티프로세싱은 각 프로세스가 독립된 메모리 공간을 가지며 무거운 프로세스 생성과 IPC를 통해 병렬 처리한다.
prompt_tokens: 25


**출력 해석:** `답:` 과 `prompt_tokens` 숫자가 나오면 1턴 호출이 정상입니다. 이 토큰 수가 멀티턴(1-3)과 비교할 baseline 입니다.

### Session 1-3. 멀티턴 루프

**하는 일:** `turn_cfg["turns"]` 3개를 순서대로 보내되, 매 턴 assistant 답을 `messages` 에 쌓아 **이전 대화를 기억한 채** 다음 턴을 이어갑니다.

**정상:** `turn 1~3 prompt_tokens=` 가 턴마다 늘어납니다.

**의미:** LLM 은 상태가 없어서, "대화를 기억한다"는 건 매 턴 *지난 대화 전체를 다시 입력으로 보낸다*는 뜻입니다. 그래서 턴이 쌓일수록 `prompt_tokens` 가 증가합니다 — 멀티턴 비용·컨텍스트 한도를 이해하는 핵심 지점입니다.

In [4]:
turns = []
usage_total = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
messages = []
for i, user_text in enumerate(turn_cfg["turns"], 1):
    messages.append(exaone.llm.ExaoneMessage(role="user", content=user_text))
    resp = client.chat(messages, options=OPT)
    answer = (resp.content or "").strip()
    messages.append(exaone.llm.ExaoneMessage(role="assistant", content=answer))
    for k in usage_total:
        usage_total[k] += int((resp.usage or {}).get(k, 0))
    pt = (resp.usage or {}).get("prompt_tokens")
    turns.append({"turn": i, "user": user_text, "assistant": answer, "usage": dict(resp.usage or {})})
    print(f"turn {i} prompt_tokens={pt}:", _preview(answer, 80))
prompt_seq = [(t["usage"] or {}).get("prompt_tokens", 0) for t in turns]
growing = len(prompt_seq) >= 2 and all(prompt_seq[i] < prompt_seq[i + 1] for i in range(len(prompt_seq) - 1))
print("누적 토큰:", usage_total)


turn 1 prompt_tokens=25: 멀티스레딩은 하나의 프로세스 내에서 스레드를 공유 메모리로 실행하고, 멀티프로세싱은 각각의 프로세스가 독립된 메모리 공간을 가지며 병렬로 실행한…
turn 2 prompt_tokens=84: CPU-bound 작업에는 멀티프로세싱이 더 적합한데, GIL로 인해 멀티스레딩은 파이썬에서 진정한 병렬 처리가 불가능하기 때문이다.
turn 3 prompt_tokens=134: C 확장 함수 사용 및 `time.sleep()`과 같은 I/O 대기 상황에서 GIL이 해제된다.
누적 토큰: {'prompt_tokens': 243, 'completion_tokens': 84, 'total_tokens': 327}


**출력 해석:** `turn N prompt_tokens=` 가 턴마다 증가하면, 지난 대화가 매번 입력에 다시 실려 컨텍스트가 누적되는 것입니다. 이게 멀티턴의 토큰 비용입니다.

### Session 1-4. 스트리밍

**하는 일:** 1-3 멀티턴 대화에 이어 스트리밍용 질문(`turn_cfg["stream_user"]`)을 `chat_stream()` 으로 보냅니다.

**정상:** `답변(stream):` 뒤로 글자가 이어서 찍히고, `첫 토큰 …ms` 가 출력됩니다.

**의미:** 스트리밍은 답 전체를 기다리지 않고 토큰을 받는 대로 흘려, 체감 지연을 줄입니다. `첫 토큰 …ms`(TTFT)가 그 핵심 지표 — 전체 생성 시간과 별개로 "첫 글자까지 걸린 시간"입니다.

In [5]:
stream_text, stream_first_ms, stream_total_ms = [], None, 0.0
stream_msgs = list(messages) + [
    exaone.llm.ExaoneMessage(role="user", content=turn_cfg["stream_user"]),
]
print("질문(stream_user):", turn_cfg["stream_user"])
started = time.monotonic()
print("답변(stream):", end=" ")
for ch in client.chat_stream(stream_msgs, options=OPT):
    if ch.kind == "text":
        if stream_first_ms is None:
            stream_first_ms = (time.monotonic() - started) * 1000
        stream_text.append(ch.text)
        print(ch.text, end="", flush=True)
stream_total_ms = (time.monotonic() - started) * 1000
print(f"\n첫 토큰 {stream_first_ms:.0f}ms / 총 {stream_total_ms:.0f}ms")


질문(stream_user): numpy 가 GIL 영향을 덜 받는 이유를 네 문장으로.
답변(stream): 

NumPy는 핵심 연산이 C 또는 포트란으로 구현되어 있다.  
이 C 코드는 계산 중에 GIL을 해제하고 실제 병렬 처리를 수행한다.  
따라서 CPU 바운드 연산에서도 멀티스레딩의 제약을 피할 수 있다.  
결과적으로 NumPy는 GIL의 영향을 덜 받는다.
첫 토큰 289ms / 총 913ms


**출력 해석:** 글자가 이어서 흐르고 `첫 토큰 …ms` 가 보이면 스트리밍이 동작하는 것입니다. 첫 토큰 시간이 사용자 체감 응답 속도를 좌우합니다.

### Session 1-5. 저장

**하는 일:** 턴별 기록과 스트림 결과를 `multi_turn.json` 으로 저장합니다.

**정상:** `saved: multi_turn.json` 이 출력됩니다.

**의미:** 토큰 추이·스트리밍 기록을 파일로 남겨 나중에 다시 확인합니다.

In [6]:
(out_dir / "multi_turn.json").write_text(json.dumps({
    "model": client.model,
    "turns": turns,
    "usage_total": usage_total,
    "streaming": {"text": "".join(stream_text), "first_token_ms": stream_first_ms, "total_ms": stream_total_ms},
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", (out_dir / "multi_turn.json").resolve())


saved: <cookbook-root>/recipes/track01_exaone_foundation/_out/multi_turn.json


**출력 해석:** `saved: multi_turn.json` 이 보이면 Session 1 기록이 저장된 것입니다.

## Session 2. 한국어 프롬프팅

**테스트 시나리오** — `golden_seed` (`korean_golden_seed.jsonl`) 10건 입니다.  
카테고리(**톤·반말·코드스위칭·함정**) 별로 다르게 적용된 system/user propmt로 각각 실행해 봅니다. 

| 카테고리 | 예시 id | 무엇을 보는지 |
|---|---|---|
| `tone` | tone-01, tone-02 | 존댓말 vs 반말 재작성 |
| `code_switch` | cs-01, cs-02 | 영어 기술 용어 유지 |
| `business` | biz-01, biz-02 | 고객·요약 스타일 |
| `arithmetic_trap` | trap-arith-* | 짧은 산술 함정 |
| `time_trap` | trap-time-* | 날짜·시각 함정 |

질문을 바꾸려면 JSONL 을 수정한 뒤 Session 2-1 부터 다시 실행하세요.


### Session 2-1. golden_seed 로드 · 1건 실행

**하는 일:** `golden_seed` 10건을 읽고, 그 중 첫번째 건을 예시로 실행해 입력·출력을 봅니다.

**입력:** `data/korean_golden_seed.jsonl`

**정상:** `시나리오:` + 10건 목록 + `id:` / `user:` / `actual:`

**의미:** Session 2-2 가 같은 `golden_seed` 전체를 돌립니다. 여기서 카테고리·질문을 먼저 확인하세요.


In [7]:
# (en) golden_seed — 10 Korean prompting fixtures; snapshot only, no pass/fail here.
# (kr) golden_seed — 한국어 프롬프팅 fixture 10건. 스냅샷만, 여기서 채점하지 않음.
golden_seed = [json.loads(line) for line in (DATA / "korean_golden_seed.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
print("시나리오: 한국어 프롬프팅 골든 시드", len(golden_seed), "건 (korean_golden_seed.jsonl)")

for item in golden_seed:
    print(_bold(f"id: {item['id']}", color="blue"))
    print("    category:", item["category"])    
    if item.get("system"):
        print("    system:", _preview(item["system"]))
    print("    user:", _preview(item["user"]))

demo = golden_seed[0]
print("\n[실행 예시]", demo['id'])
OPT_KO = exaone.llm.ExaoneGenerateOptions(enable_thinking=False, max_new_tokens=384)
msgs = [exaone.llm.ExaoneMessage(role="user", content=demo["user"])]
if demo.get("system"):
    msgs.insert(0, exaone.llm.ExaoneMessage(role="system", content=demo["system"]))
resp = client.chat(msgs, options=OPT_KO)
actual = (resp.content or "").strip()
if demo.get("system"):
    print("system:", _preview(demo["system"], 100))
print("user:", demo["user"])
print("actual:", actual)

시나리오: 한국어 프롬프팅 골든 시드 10 건 (korean_golden_seed.jsonl)
id: tone-01
    category: tone
    system: You are a Korean-speaking assistant. Always answer in p…
    user: 동료에게 회의가 30분 미뤄졌다고 알리는 카톡 한 문장을 써줘.
id: tone-02
    category: tone
    system: You rewrite Korean sentences into a friendly 반말 tone fo…
    user: 예시:
  IN : 일정 변경 가능하실까요?
  OUT: 일정 좀 바꿔도 돼?
  IN : 자료 공…
id: cs-01
    category: code_switch
    system: You are a Korean-speaking technical writer. Answer in K…
    user: Kubernetes 에서 sidecar 컨테이너의 용도를 한 문단으로 설명해줘. Pod 와의 관계도…
id: cs-02
    category: code_switch
    system: You are a Korean-speaking ML engineer assistant. Answer…
    user: fine-tuning 과 prompt engineering 의 차이를 두 문장으로 설명해줘.
id: biz-01
    category: business
    system: You are a Korean customer support agent. Respond polite…
    user: 환불은 며칠 안에 처리되나요?
id: biz-02
    category: business
    system: You are a Korean executive summary writer. Compress the…
    user: 오늘 오전 회의에서 신규 프로젝트 일정이 다음 주로 연기되었고, 기획팀에서 PRD 초안을 이번

**출력 해석:** `시나리오:` 목록과 예시 1건의 한글 답변(`actual`)이 보이면, `golden_seed` 로드와 예시 실행이 된 것입니다.

### Session 2-2. 시드 10건 실행

**하는 일:** `golden_seed` 10건을 같은 방식으로 호출해 카테고리별 답변을 모읍니다.

**정상:** id 10줄과 각 답변(actual)이 출력됩니다.

**의미:** 톤·반말·코드스위칭·함정 카테고리에서 프롬프트가 의도대로 먹는지 한 번에 훑는 단계입니다.

In [8]:
golden_rows = []
for item in golden_seed:
    msgs = [exaone.llm.ExaoneMessage(role="user", content=item["user"])]
    if item.get("system"):
        msgs.insert(0, exaone.llm.ExaoneMessage(role="system", content=item["system"]))
    resp = client.chat(msgs, options=OPT_KO)
    actual = (resp.content or "").strip()
    row = dict(item)
    row["actual"] = actual
    golden_rows.append(row)
    print(_bold(item["id"], color="blue"), _preview(actual, 60))


tone-01 회의가 30분 미뤄졌어요. 확인해 주세요.
tone-02 회의 자료는 붙였으니 확인하고 의견 줘.
cs-01 Kubernetes에서 sidecar 컨테이너는 동일한 Pod 내 주 애플리케이션 컨테이너와 함께 실행되어 …
cs-02 Fine-tuning은 모델 파라미터를 업데이트해 특정 작업에 맞게 조정하는 것이고, prompt engin…
biz-01 환불은 일반적으로 결제 후 3~7영업일 이내에 처리됩니다. 확인 후 다시 안내드리겠습니다.
biz-02 - 신규 프로젝트 일정은 다음 주로 연기  
- 기획팀, 금주 금요일까지 PRD 초안 공유  
- 디자인팀,…
trap-arith-01 9마리
trap-arith-02 그 한 명의 남은 사과는 1.5개입니다.
trap-time-01 6월 1일 화요일입니다.
trap-time-02 회의 시작 시각은 오후 3시입니다.


**출력 해석:** id 10건의 답변이 보이면 시드 전체 실행이 된 것입니다.

### Session 2-3. JSONL 저장

**하는 일:** 10건 결과를 `korean_golden.jsonl` 로 저장합니다.

**정상:** `(10 rows)` 가 출력됩니다.

**의미:** 한 줄에 JSON 1건(JSONL) 형식 — 스트리밍 처리·증분 추가에 편한 포맷입니다.

In [9]:
path = out_dir / "korean_golden.jsonl"
with path.open("w", encoding="utf-8") as f:
    for row in golden_rows:
        f.write(json.dumps({"model": client.model, **row}, ensure_ascii=False) + "\n")
print("saved:", path.resolve(), f"({len(golden_rows)} rows)")


saved: <cookbook-root>/recipes/track01_exaone_foundation/_out/korean_golden.jsonl (10 rows)


**출력 해석:** `(10 rows)` 가 보이면 10건이 JSONL 로 저장된 것입니다.

## Session 3. 구조화 출력

**테스트 시나리오** — 크게 두 갈래입니다. **A** 는 API 사용 없이 깨진 JSON 복구하는 유틸리티를 사용합니다.  
**B** 는 회의록을 입력으로하여 JSON 형식의 structured_output 답변을 얻습니다. 이때 회의록 내용에 대해 원하는 형식의 `action_items` 이 산출되도록 structured_output의 schema를 지정하여 수행합니다.

| 데이터 | Session | 내용 |
|---|---|---|
| `structured_output_samples.json` | 3-1~3-2 | 깨진 JSON 3건 (fence, trailing comma) |
| `meeting_minutes_samples.json` + `minutes_cfg` | 3-3~3-4 | 회의록 5건 → action_items |

**파싱 OK/FAIL** 과 `items≥1` 여부를 확인합니다. (답변 완성도와 관계없이 기능 작동 여부를 중점적으로 확인)


### Session 3-1. samples 로드 (오프라인 JSON 복구)

**하는 일:** `samples` — 깨진 JSON 3건을 읽고 `StructuredOutputPipeline` 을 만듭니다.

**입력:** `data/structured_output_samples.json`

**정상:** `시나리오:` + `샘플 수: 3` + id 3줄 (`clean`, `fenced`, `trailing_comma`)

**의미:** Session 3-2 가 이 `samples` 를 `process()` 합니다. API 키 없이도 통과 가능합니다.


In [10]:
person_schema = {
    "type": "object", "required": ["name", "age"],
    "properties": {"name": {"type": "string"}, "age": {"type": "integer", "minimum": 0, "maximum": 150}},
}
pipeline = exaone.output.StructuredOutputPipeline(json_schema=person_schema, max_repair_attempts=1)
# (en) Offline broken-JSON repair fixtures (no API).
# (kr) API 없이 깨진 JSON 복구용 fixture.
samples = json.loads((DATA / "structured_output_samples.json").read_text(encoding="utf-8"))
print("시나리오: 깨진 JSON 복구", len(samples), "건 (structured_output_samples.json)")
for s in samples:
    print(_bold(s['id']))
    print(f"{s['raw']}")


시나리오: 깨진 JSON 복구 3 건 (structured_output_samples.json)
clean
{"name":"홍길동","age":30}
fenced
결과:
```json
{"name":"홍길동","age":30}
```
trailing_comma
{"name":"홍길동","age":30,}


**출력 해석:** `시나리오:` 와 3가지 샘플이 보이면 로드 성공한 것입니다.


### Session 3-2. 깨진 JSON 복구

**하는 일:** 깨진 샘플 3건(clean·fenced·trailing_comma)을 `pipeline.process()` 로 복구합니다.

**정상:** 3줄 모두 OK 가 출력됩니다.

**의미:** 모델은 *거의 맞지만 정확하지 않은* JSON 을 생성할 수 있습니다. `StructuredOutputPipeline` 은 이런 깨짐을 규칙으로 복구해, API 없이도 파싱 성공률을 끌어올립니다.

In [11]:
oks = 0
for sample in samples:
    r = pipeline.process(sample["raw"])
    ok = r.success
    oks += int(ok)
    print(_bold(sample["id"]), "OK" if ok else "FAIL", r.data if ok else r.error)


clean OK {'name': '홍길동', 'age': 30}
fenced OK {'name': '홍길동', 'age': 30}
trailing_comma OK {'name': '홍길동', 'age': 30}


**출력 해석:** 3줄 모두 OK 면 펜스·끝쉼표 같은 흔한 깨짐이 복구된 것입니다. 모델 출력 파싱의 1차 방어선입니다.

### Session 3-3. minutes_cfg 로드 (회의록 API)

**하는 일:** `minutes_cfg` · `minutes_samples` · `ACTION_ITEM_SCHEMA` 를 읽고 API용 `opt_json` 을 만듭니다.

**입력:** `action_item_schema.json`(출력양식), `minutes_prompt.json`(지시문), `meeting_minutes_samples.json`(회의록)

**정상:** `시나리오:` + `회의록 샘플: 5` + m01~m05 제목 한 줄씩

**의미:** Session 3-4에서 사용할 `minutes_cfg` · `minutes_samples` 5건을 준비합니다.


In [35]:
ACTION_ITEM_SCHEMA = json.loads((DATA / "action_item_schema.json").read_text(encoding="utf-8"))
minutes_pipeline = exaone.output.StructuredOutputPipeline(json_schema=ACTION_ITEM_SCHEMA, max_repair_attempts=1)
minutes_cfg = json.loads((DATA / "minutes_prompt.json").read_text(encoding="utf-8"))
prefix = minutes_cfg.get("user_prefix", "")
minutes_samples = json.loads((DATA / "meeting_minutes_samples.json").read_text(encoding="utf-8"))
print("시나리오: 회의록 → action_items", len(minutes_samples), "건 (meeting_minutes_samples.json)")
for s in minutes_samples:
    print(f"  {s['id']} {s['title']}:", _preview(s['minutes'], 50))
response_format = {"type": "json_schema", "json_schema": {"name": "action_items", "schema": ACTION_ITEM_SCHEMA}}
opt_json = exaone.llm.ExaoneGenerateOptions(enable_thinking=False, max_new_tokens=1024, response_format=response_format, frequency_penalty=0.8)
minutes_rows = []
print("회의록 샘플:", len(minutes_samples))

시나리오: 회의록 → action_items 5 건 (meeting_minutes_samples.json)
  m01 주간 백엔드 동기화: 주간 백엔드 동기화 회의 — 5/28(목) 10시. 이서연은 결제 모듈의 idempoten…
  m02 프로덕트 위클리: 프로덕트 위클리 — 김프로 PM 주관. 신규 온보딩 플로우 와이어프레임은 디자인팀이 다음 …
  m03 보안 점검: 보안 점검 회의 — 위협 시나리오 6건 식별. 이중 prompt injection 1건이 …
  m04 데이터 거버넌스: 데이터 거버넌스 분기 회의. PII 분류 정책 v2 초안을 데이터팀 최영주가 다음 주 수요…
  m05 AX 추진 킥오프: AX 추진 킥오프 — 조직 KB QA 봇 PoC 를 시작. 김민수는 다음 주 금요일까지 R…
회의록 샘플: 5


**출력 해석:** `시나리오:` 와 m01~m05 제목이 보이면 `minutes_cfg` · `minutes_samples` 가 준비된 것입니다.


### Session 3-4. API 호출·검증

**하는 일:** 회의록 5건을 모델에 보내 `ACTION_ITEM_SCHEMA` 형식의 structured output 을 받고, 파이프라인으로 파싱·검증합니다.

**정상:** `m01 OK items≥1` 처럼 대부분 OK 가 나옵니다.

**의미:** 스키마를 주면 모델이 자유 문장이 아니라 *정해진 JSON 구조*로 답하도록 강제됩니다. OK = `action_items` 구조 파싱 성공 — 회의록처럼 비정형 입력에서 구조화 데이터를 뽑는 전형적 패턴입니다.

In [39]:
for s in minutes_samples:
    resp = client.chat([exaone.llm.ExaoneMessage(role="user", content=prefix + s["minutes"])], options=opt_json)
    raw = resp.content or resp.reasoning_content or ""
    parsed = minutes_pipeline.process(raw.strip())
    n = len(parsed.data.get("action_items", [])) if parsed.success else 0
    minutes_rows.append({"id": s["id"], "success": parsed.success, "data": parsed.data, "error": parsed.error})
    if parsed.success:
        print(s["id"], "OK", f"items={n}")
    else:
        print(s["id"], "FAIL")
        print("  raw:", _preview(raw, 200))
        if parsed.error:
            print("  error:", parsed.error, "len chars:", len(raw))
ok = sum(1 for r in minutes_rows if r["success"])


m01 OK items=3
m02 OK items=3
m03 OK items=2
m04 OK items=2
m05 OK items=2


**출력 해석:** `OK items≥1` 이면 회의록에서 원하는 `action_items` 구조가 스키마대로 추출된 것입니다.

### Session 3-5. 저장

**하는 일:** 회의록 결과와 성공 수를 `action_items.json` 으로 저장합니다.

**정상:** `saved:` 와 `(N/5)` 가 출력됩니다.

**의미:** 성공/전체(N/5)를 함께 남겨, 구조화 출력 성공률을 회귀로 추적합니다.

In [14]:
(out_dir / "action_items.json").write_text(json.dumps({
    "model": client.model, "stats": {"total": len(minutes_rows), "success": ok},
    "samples": minutes_rows,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", (out_dir / "action_items.json").resolve(), f"({ok}/{len(minutes_rows)})")


saved: <cookbook-root>/recipes/track01_exaone_foundation/_out/action_items.json (5/5)


**출력 해석:** `saved:` 와 `(N/5)` 가 보이면 5건 중 성공 건수와 함께 저장된 것입니다.

## Session 4. 함수 호출

**테스트 시나리오** — 환산을 묻는 질문에 대해 모델이 고정 환율표를 조회하는 도구를 호출하고(turn 1), 도구 결과를 활용하여 답변(turn 2)하는지 확인 합니다.

| 데이터 | Session | 사용 내용 |
|---|---|---|
| `exchange_rates.json` | 4-1 | 로컬 `exchange_rate_tool()` 검증 |
| `fc_demo.json` → `user` | 4-2~4-3 | 「100달러를 원화로」→ tool_calls → 최종 답 |

`quote: KRW` 로 호출되는지, turn2 에 원화 답이 오는지 확인합니다.


### Session 4-1. 로컬 환율 도구

**하는 일:** 환율 조회 도구 `exchange_rate_tool()` 을 정의하고, `exchange_rates.json` 의 고정 환율표로 직접 호출해 봅니다.

**입력:** `exchange_rates.json`

**정상:** `rate` 숫자가 출력됩니다.

**의미:** 모델에 붙이기 전에 도구 자체가 제대로 도는지 먼저 확인합니다 — 함수 호출은 "모델이 부를 함수"가 먼저 정상이어야 합니다.

In [15]:
_rates_doc = json.loads((DATA / "exchange_rates.json").read_text(encoding="utf-8"))
RATES_KRW = _rates_doc["rates"]
CURRENCIES = sorted(set(RATES_KRW) | {"KRW"})

def exchange_rate_tool(base: str, quote: str = "KRW") -> dict:
    # (en) Lookup rates from data/exchange_rates.json.
    # (kr) data/exchange_rates.json 표를 조회한다.
    base, quote = base.upper(), quote.upper()
    table = set(RATES_KRW) | {"KRW"}
    if base not in table or quote not in table:
        return {"error": f"unknown: {base}/{quote}"}
    if base == quote:
        rate = 1.0
    elif base == "KRW":
        rate = 1.0 / RATES_KRW[quote]
    elif quote == "KRW":
        rate = RATES_KRW[base]
    else:
        rate = RATES_KRW[base] / RATES_KRW[quote]
    return {"base": base, "quote": quote, "rate": round(rate, 4), "as_of": _rates_doc["as_of"]}

demo_rate = exchange_rate_tool("USD")
print("USD → KRW:", demo_rate)


USD → KRW: {'base': 'USD', 'quote': 'KRW', 'rate': 1380.5, 'as_of': '2026-05-28'}


**출력 해석:** `rate` 숫자가 나오면 도구 자체는 정상입니다. 이제 모델이 이걸 호출하게 하면 됩니다(4-2).

### Session 4-2. 턴1 — fc_demo + tool_calls

**하는 일:** `fc_demo["user"]` 질문으로 모델이 `exchange_rate` 를 **함수 호출**로 요청하는지 봅니다.

**입력:** `data/fc_demo.json`

**정상:** `fc_demo user:` + `turn1 tool_calls: True` + `quote: KRW`

**의미:** 모델이 임의로 답하지 않고 목적에 맞게 함수 호출을 하는지 여부와 도구 결과를 확인합니다.


In [16]:
EXCHANGE_TOOL = {
    "type": "function",
    "function": {
        "name": "exchange_rate",
        "description": "Get exchange rate (use quote=KRW for Korean won).",
        "parameters": {
            "type": "object", "required": ["base", "quote"],
            "properties": {
                "base": {"type": "string", "enum": CURRENCIES},
                "quote": {"type": "string", "enum": CURRENCIES},
            },
        },
    },
}
fc_demo = json.loads((DATA / "fc_demo.json").read_text(encoding="utf-8"))
print("fc_demo user:", fc_demo["user"])
msgs = [exaone.llm.ExaoneMessage(role="user", content=fc_demo["user"])]
opt_fc = exaone.llm.ExaoneGenerateOptions(enable_thinking=False, max_new_tokens=256, tools=[EXCHANGE_TOOL])
r1 = client.chat(msgs, options=opt_fc)
print("turn1 tool_calls:", bool(r1.tool_calls))
executed = []
if r1.tool_calls:
    for call in r1.tool_calls:
        fn = call.get("function") or {}
        args = json.loads(fn.get("arguments") or "{}")
        result = exchange_rate_tool(**args) if fn.get("name") == "exchange_rate" else {"error": "unknown"}
        executed.append({"tool_call_id": call.get("id"), "name": fn.get("name"), "arguments": args, "result": result})
        print(" executed:", fn.get("name"), args, "→", result)
krw_ok = bool(executed) and executed[0]["arguments"].get("quote") == "KRW"


fc_demo user: 100달러를 원화(KRW)로 환산해줘.


turn1 tool_calls: True
 executed: exchange_rate {'base': 'USD', 'quote': 'KRW'} → {'base': 'USD', 'quote': 'KRW', 'rate': 1380.5, 'as_of': '2026-05-28'}


**출력 해석:** `turn1 tool_calls: True` 와 `quote: KRW` 가 보이면, 모델이 직접 답을 지어내지 않고 환율 도구를 호출하기로 결정한 것입니다.

### Session 4-3. 턴2 — 최종 답변·저장

**하는 일:** 도구 결과를 `messages` 에 다시 넣어 모델이 최종 답을 만들게 하고, 호출 trace 를 저장합니다.

**정상:** `turn2:` 에 원화 답, `saved:` 가 출력됩니다.

**의미:** 함수 호출은 *호출→결과 주입→재호출*의 2턴 루프입니다. 모델이 도구가 돌려준 실제 숫자에 근거해 답하는지가 핵심입니다.

In [17]:
tool_trace = {}
r2 = None
if r1.tool_calls:
    msgs2 = list(msgs) + [exaone.llm.ExaoneMessage(role="assistant", content=r1.content or "", tool_calls=r1.tool_calls)]
    for ex in executed:
        msgs2.append(exaone.llm.ExaoneMessage(role="tool", tool_call_id=ex["tool_call_id"], name=ex["name"],
                                              content=json.dumps(ex["result"], ensure_ascii=False)))
    r2 = client.chat(msgs2, options=opt_fc)
    print("turn2:", r2.content)
    tool_trace = {"turn_1": {"tool_calls": r1.tool_calls}, "tool_execution": executed, "turn_2": {"content": r2.content}}
    (out_dir / "tool_loop_trace.json").write_text(json.dumps(tool_trace, ensure_ascii=False, indent=2), encoding="utf-8")
    print("saved:", (out_dir / "tool_loop_trace.json").resolve())
ok_fc = r1.tool_calls and r2 and (r2.content or "").strip()


turn2: 현재 환율은 1 USD = 1,380.5 KRW입니다.

따라서 100달러는:
**100 USD × 1,380.5 KRW/USD = 138,050 KRW** 입니다.

100달러는 약 **138,050원**에 해당합니다.
saved: <cookbook-root>/recipes/track01_exaone_foundation/_out/tool_loop_trace.json


**출력 해석:** `turn2:` 에 원화 답과 `saved:` 가 보이면, 모델이 도구 결과를 근거로 최종 답을 만든 것입니다.

## Session 5. ThinkingRouter

**테스트 시나리오** — `router_inputs` (`router_queries.json`) 5건입니다. 잡담·산술·코드·JSON·RAG 후속 등 **유형**이 포함되어 있습니다.

| id | 질문 요지 | 힌트 플래그 |
|---|---|---|
| `chitchat` | 인사 한 문장 | 단순 대화 |
| `arith` | 기차 만남 시각 | thinking 켜짐 흔함 |
| `code_qa` | 버그 있는 함수 | 코드 설명 |
| `extract_json` | JSON 추출 | `force_json` |
| `rag_followup` | 검색 결과 요약 | `has_context` |

의도 라벨·thinking 옵션이 질문 유형에 맞게 바뀌는지 봅니다.


### Session 5-1. router_inputs 로드

**하는 일:** `router_inputs` 5건과 `ThinkingRouter` 를 준비합니다.

**입력:** `data/router_queries.json`

**정상:** `시나리오:` + `질문 수: 5` + id·query 미리보기 5줄

**의미:** Session 5-2 가 각 행에 대해 `route()` 후 `chat()` 합니다.


In [18]:
router_inputs = json.loads((DATA / "router_queries.json").read_text(encoding="utf-8"))
router = exaone.agents.ThinkingRouter(client=client, model=client.model)
route_rows = []
print("시나리오: ThinkingRouter 질문", len(router_inputs), "건 (router_queries.json)")
for item in router_inputs:
    flags = []
    if item.get("has_context"):
        flags.append("has_context")
    if item.get("has_tools"):
        flags.append("has_tools")
    if item.get("force_json"):
        flags.append("force_json")
    print(_bold(item['id']), ":", _preview(item['query'], 50), "[" + ", ".join(flags) + "]" if flags else "")


시나리오: ThinkingRouter 질문 5 건 (router_queries.json)
chitchat : 안녕! 한 문장으로 답해줘. 
arith : 기차 A 60km/h 9시 출발, B 90km/h 9:30 출발. 만나는 시각을 계산해줘. 
code_qa : 이 함수 버그 설명:
def avg(nums):
    s=0
    for n in nu… 
extract_json : 회사명/직책/연봉만 JSON 으로: 홍길동은 ACME 백엔드, 연봉 7천만원. [force_json]
rag_followup : 위 검색 결과로 한국 휴일 정책을 세 줄로 요약해줘. [has_context]


**출력 해석:** `시나리오:` 와 5개 유형의 질문이 보이면 `router_inputs` 가 로드된 것입니다.

### Session 5-2. 라우팅·실행·저장

**하는 일:** 5건 각각을 `router.route()` 로 의도 판정한 뒤 `chat()` 하고, 결과를 JSON 으로 저장합니다.

**정상:** 5줄에 `의도=` 와 `답:` 이 출력됩니다.

**의미:** ThinkingRouter 는 질문 유형에 따라 thinking(추론) 사용 여부를 정합니다. 잡담은 끄고, 산술·코드처럼 단계적 추론이 필요한 질문은 켜는 식 — 쉬운 질문에 과한 추론 비용을 안 쓰게 하는 라우팅입니다.

In [19]:
for item in router_inputs:
    d = router.route(item["query"], history=None, has_tools=item.get("has_tools", False), has_context=item.get("has_context", False))
    resp_format = {"type": "json_object"} if item.get("force_json") else None
    opt = exaone.llm.ExaoneGenerateOptions(
        enable_thinking=d.enable_thinking, temperature=d.temperature, top_p=d.top_p,
        max_new_tokens=384, response_format=resp_format,
    )
    thinking_label = "켜짐" if d.enable_thinking else "꺼짐"
    try:
        resp = client.chat([exaone.llm.ExaoneMessage(role="user", content=item["query"])], options=opt)
        answer = (resp.content or resp.reasoning_content or "").strip()
        print(_bold(item['id']), f": 의도={d.semantic_intent}, thinking={thinking_label}")
        print("답:", _preview(answer, 100))
        route_rows.append({
            "id": item["id"], "intent": d.semantic_intent, "thinking": d.enable_thinking,
            "content": answer[:120], "error": None,
        })
    except Exception as exc:
        # (en) force_json / long prompts can return a non-object API payload; log and continue.
        # (kr) force_json·긴 프롬프트는 API가 비정상 payload 를 줄 수 있어 로그만 남기고 계속한다.
        print(_bold(item['id']), f": 의도={d.semantic_intent}, thinking={thinking_label} — chat 실패 ({type(exc).__name__})")
        route_rows.append({
            "id": item["id"], "intent": d.semantic_intent, "thinking": d.enable_thinking,
            "content": "", "error": str(exc),
        })

(out_dir / "route_comparison.json").write_text(json.dumps({"rows": route_rows}, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", (out_dir / "route_comparison.json").resolve())


chitchat : 의도=general, thinking=꺼짐
답: 안녕하세요! 😊
arith : 의도=analytical, thinking=켜짐
답: 기차 A와 기차 B가 같은 방향으로 이동한다고 가정하고, 기차 B가 기차 A를 따라잡는 시각을 계산해 보겠습니다.

---

### 주어진 정보:

- 기차 A: 속도 = 60 k…
code_qa : 의도=analytical, thinking=켜짐
답: 주어진 함수 `avg(nums)`에는 **중요한 버그**가 있습니다. 아래에서 문제를 분석하고 수정 방법을 제시하겠습니다.

---

### 🔍 함수 분석

```python
de…
extract_json : 의도=structured, thinking=꺼짐
답: {
  "회사명": "ACME",
  "직책": "백엔드",
  "연봉": 70000000
}
rag_followup : 의도=analytical, thinking=켜짐
답: 한국의 휴일 정책은 법정 공휴일과 대체 휴일 제도를 포함한다.  
공휴일이 주말과 겹칠 경우 다음 평일을 대체 휴일로 지정한다.  
정부는 사회적 합의와 경제 상황을 고려해 휴일 …
saved: <cookbook-root>/recipes/track01_exaone_foundation/_out/route_comparison.json


**출력 해석:** 유형마다 `의도=` 와 `답:` 이 보이면 라우팅이 동작하는 것입니다. 산술·코드 유형에서 thinking 이 켜지는 경향을 확인하세요.

## 체크포인트


- [ ] Session 1 `multi_turn.json`
- [ ] Session 2 `korean_golden.jsonl` (10 rows)
- [ ] Session 3 `action_items.json`
- [ ] Session 4 `tool_loop_trace.json`
- [ ] Session 5 `route_comparison.json`

**다음:** Track 02 — Minimum Agent Loop
